# 자연어 처리 7일차 (2) — 문장 유사도로 영화 추천 : CountVectorizer vs SentenceTransformer

수업 필기 `nlp_실습6_영화추천서비스`(Day 041 `nlp_실습6` 의 수정본)를 정리하고, 교재 **p.117 '문장 유사도를 이용한 추천시스템'** 대로 줄거리 벡터를 **SentenceTransformer** 로 바꿔 비교한 노트입니다.

- CountVectorizer 버전의 자세한 정리(희소 행렬 · 자기 자신 제외 · 같은 제목 처리)는 [Day 041 `movie_recommendation.ipynb`](../../2026-11-05/lecture/movie_recommendation.ipynb) 에 있습니다.
- 원본에서 고친 부분은 **📝 고친 부분** 으로 적었습니다.

## 0. 설치 · 실행 방법

| 패키지 | 이 노트에서 쓰는 것 | 설치 |
|---|---|---|
| `pandas` · `scikit-learn` | 데이터 · `CountVectorizer` · `cosine_similarity` | Colab 기본 포함 |
| `sentence-transformers` | 줄거리 문장 임베딩 (`all-MiniLM-L12-v2`, 133MB, 첫 실행 때 자동 다운로드 · **인터넷 필요**) | Colab 기본 포함 · 로컬은 `pip install sentence-transformers` |

**데이터** — Kaggle The Movies Dataset `movies_metadata.csv`. 노트북 폴더에 원본이 있으면 그것을, 없으면 [Day 041 폴더의 `data/movies_overview.csv.gz`](../../2026-11-05/lecture/data/)(필요한 5개 열만, 7MB)를 읽는다.

**실행 순서 · 시간** : 맨 위부터 **[런타임] → [모두 실행]**. 줄거리 1만 개 임베딩이 CPU 로 약 3~5분(GPU 면 수십 초).

In [1]:
# ① 설치 (로컬에서 처음 한 번 — Colab 은 기본 포함)
# !pip install -q sentence-transformers scikit-learn pandas

In [2]:
import warnings, os
warnings.filterwarnings('ignore')
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'      # 다운로드 진행 막대 숨기기
from transformers.utils import logging as hf_logging
hf_logging.set_verbosity_error(); hf_logging.disable_progress_bar()
from huggingface_hub.utils import logging as hub_logging
hub_logging.set_verbosity_error()                     # 'unauthenticated requests' 안내 숨기기 (공개 모델은 로그인 없이 받아진다)

In [3]:
import numpy as np
import pandas as pd
for path in ['movies_metadata.csv', 'data/movies_overview.csv.gz', 'movies_overview.csv.gz',
             '../../2026-11-05/lecture/data/movies_overview.csv.gz']:
    if os.path.exists(path):
        break
df = pd.read_csv(path, low_memory=False)
df2 = df[['overview', 'title']].dropna().reset_index(drop=True)
print(path, '| 결측 제거 후', len(df2), '편')

df3 = df2[0:10000].reset_index(drop=True)                 # 원본과 같은 앞 1만 편
print('사용 :', len(df3), '편')

movies_overview.csv.gz | 결측 제거 후 44506 편
사용 : 10000 편


> **📝 고친 부분 (원본 · Day 041 에서 이어서)** 수정본은 `df2[0:10000]` 으로 **1만 편**이 맞게 바뀌었다(Day 041 필기는 `[0:9999]` = 9,999편). `drive.mount` · `DtypeWarning` 은 Day 041 과 같이 파일 경로 하나 · `low_memory=False` 로.

## 1. CountVectorizer — 단어가 겹치는 정도

In [4]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

cv = CountVectorizer(stop_words='english')
X_count = cv.fit_transform(df3['overview'])               # 희소 행렬 그대로 (toarray() 하지 않음)
print('행렬 :', X_count.shape, f'| 단어 사전 {len(cv.vocabulary_):,}개')

title_idx = pd.Series(df3.index, index=df3['title'])

def recommend(title, X, k=10):
    i = title_idx[title]
    i = i.iloc[0] if isinstance(i, pd.Series) else i      # 같은 제목이 여러 편이면 첫 번째
    sims = cosine_similarity(X[i:i + 1], X).ravel()
    sims[i] = -1                                           # 자기 자신은 위치로 제외
    top = np.argsort(-sims)[:k]
    return pd.DataFrame({'title': df3['title'].values[top], 'similarity': sims[top].round(3)}, index=range(1, k + 1))

recommend('Toy Story', X_count)

행렬 : (10000, 32382) | 단어 사전 32,382개


,title,similarity
1,Toy Story 2,0.412
2,The Champ,0.245
3,Man on the Moon,0.143
4,Rivers and Tides,0.143
5,Class of 1984,0.139
6,Malice,0.131
7,Heartbeeps,0.129
8,Condorman,0.128
9,Rebel Without a Cause,0.127
10,The First $20 Million Is Always the Hardest,0.124


원본과 같이 `Toy Story 2` 0.412 · `The Champ` 0.245 … (1만 편 기준). `Andy` · `Woody` 같은 **이름이 겹치는** 영화가 끼는 문제는 Day 041 에서 봤다.

> **📝 고친 부분 (원본 cell 10~16)** `.toarray()` (1만 × 3만 2천 정수 ≈ 2.6GB) · 1만 × 1만 유사도 표(0.8GB) 두 개 → 희소 행렬 그대로 + **한 편 × 전체**만 계산. `[1:n+1]` 대신 자기 위치를 제외(동점 · 중복 영화가 있어도 안전). 없는 제목이면 `None` 대신 오류 메시지 · 같은 제목은 첫 번째.

## 2. SentenceTransformer — 뜻이 비슷한 정도 (교재 p.117)

In [5]:
import time
from sentence_transformers import SentenceTransformer
st_model = SentenceTransformer('sentence-transformers/all-MiniLM-L12-v2')
t0 = time.time()
X_st = st_model.encode(df3['overview'].tolist(), batch_size=64, normalize_embeddings=True, show_progress_bar=False)
print('문장 임베딩 :', X_st.shape, f'| {time.time() - t0:.0f}초 · 크기 {X_st.nbytes / 1e6:.1f}MB (밀집 · 0 없음)')
recommend('Toy Story', X_st)

문장 임베딩 : (10000, 384) | 203초 · 크기 15.4MB (밀집 · 0 없음)


,title,similarity
1,Toy Story 2,0.786
2,Elf,0.446
3,Pinocchio,0.428
4,Heartbeeps,0.425
5,Child's Play 3,0.424
6,Class of 1984,0.417
7,The Apartment,0.400
8,Firestarter,0.398
9,The Divorcee,0.397
10,The Fly,0.389


In [6]:
def side_by_side(title, k=10):
    a, b = recommend(title, X_count, k), recommend(title, X_st, k)
    out = pd.DataFrame({'Count 추천': a['title'], 'Count 유사도': a['similarity'],
                        '문장 임베딩 추천': b['title'], '문장 임베딩 유사도': b['similarity']})
    print(f"{title} — 두 방법 상위 {k}편 겹침 : {len(set(a['title']) & set(b['title']))}편")
    return out

side_by_side('GoldenEye')

GoldenEye — 두 방법 상위 10편 겹침 : 6편


,Count 추천,Count 유사도,문장 임베딩 추천,문장 임베딩 유사도
1,Licence to Kill,0.241,Live and Let Die,0.628
2,Never Say Never Again,0.197,Octopussy,0.618
3,You Only Live Twice,0.195,Never Say Never Again,0.606
4,Casino Royale,0.195,The Living Daylights,0.595
5,Octopussy,0.186,The Man with the Golden Gun,0.592
6,Live and Let Die,0.182,Goldfinger,0.574
7,Phantasm,0.182,Casino Royale,0.574
8,Dr. No,0.174,Licence to Kill,0.571
9,Diamonds Are Forever,0.156,Die Another Day,0.568
10,Thunderball,0.153,Dr. No,0.554


In [7]:
side_by_side('Jumanji')

Jumanji — 두 방법 상위 10편 겹침 : 0편


,Count 추천,Count 유사도,문장 임베딩 추천,문장 임베딩 유사도
1,Word Wars,0.271,Flowers in the Attic,0.535
2,Brainscan,0.255,Eight Days a Week,0.533
3,Quintet,0.246,Pete's Dragon,0.521
4,Masques,0.234,Thir13en Ghosts,0.514
5,eXistenZ,0.226,Peter Pan,0.510
6,Nirvana,0.218,Panic Room,0.509
7,The Last Starfighter,0.207,Labyrinth,0.487
8,Dungeons & Dragons,0.198,The Glass House,0.486
9,Poolhall Junkies,0.191,The Boy Who Could Fly,0.472
10,5 Card Stud,0.185,Amityville: Dollhouse,0.471


In [8]:
# 줄거리를 직접 써서 찾기 — 제목이 사전에 없어도 된다 (단어가 안 겹쳐도)
query = 'A group of toys come to life when their owner is away.'
q = st_model.encode([query], normalize_embeddings=True)
sims = (X_st @ q.T).ravel()                                # 길이 1 이라 내적 = 코사인
top = np.argsort(-sims)[:5]
print('문장 임베딩 :', [(df3['title'][i], round(float(sims[i]), 3)) for i in top])
qc = cv.transform([query])
sc = cosine_similarity(qc, X_count).ravel()
top_c = np.argsort(-sc)[:5]
print('Count       :', [(df3['title'][i], round(float(sc[i]), 3)) for i in top_c])

문장 임베딩 : [('Dolls', 0.502), ('Toy Story', 0.404), ('Pinocchio', 0.398), ('Toy Story 2', 0.389), ('Stuart Little 2', 0.366)]
Count       : [('Trapped in Paradise', 0.309), ("This Girl's Life", 0.289), ('Babylon 5: The River of Souls', 0.289), ('Possible Loves', 0.285), ('Raising Helen', 0.261)]


- CountVectorizer 는 **같은 단어**(이름 · 고유명사)가 겹쳐야 점수가 생기고, SentenceTransformer 는 **표현이 달라도 뜻이 비슷하면** 점수가 생긴다(위 직접 쓴 줄거리 검색).
- 대신 문장 임베딩은 모델이 만든 점수라 왜 추천됐는지 설명하기 어렵고, 1만 편 임베딩에 몇 분이 든다(Count 는 1초). 고유명사가 중요한 '시리즈 찾기' 는 Count 가 오히려 잘할 때도 있다 — 위 표에서 직접 비교.
- `all-MiniLM-L12-v2` 는 **최대 128 토큰**까지만 읽는다(`st_model.max_seq_length`) — 긴 줄거리는 뒷부분이 잘린다.

## 정리

| | CountVectorizer | SentenceTransformer |
|---|---|---|
| 벡터 | 희소 (1만 × 3만, 대부분 0) | 밀집 (1만 × 384) |
| 비슷함의 기준 | 같은 단어가 겹치는가 | 뜻이 비슷한가 (문장 쌍으로 학습) |
| 속도 | 빠름 | 느림 (모델 계산) |
| 약점 | 이름만 같은 영화 · 다른 표현은 0 | 긴 글 잘림 · 근거 설명 어려움 |